# ⚛️ Módulo 5: Dinámica Molecular
## Actividad 5.3: Condiciones de Contorno y Ensambles Estadísticos

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_05_dinamica_molecular/03_condiciones_contorno_ensemble.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Comprender e implementar las condiciones periódicas de contorno (PBC)
- Aplicar la convención de imagen mínima para el cálculo de distancias
- Distinguir los ensambles NVE, NVT y NPT y sus usos en biomoléculas
- Calcular propiedades termodinámicas (temperatura, presión) a partir de trayectorias
- Entender el papel de las cajas de simulación en DM

---

## 1. Instalación de Dependencias

In [ ]:
!pip install numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.constants import k as k_B

print("Bibliotecas importadas correctamente")

## 2. ¿Por qué se Necesitan Condiciones de Contorno?

Una caja de simulación típica contiene solo miles a millones de átomos. Sin condiciones de contorno:
- La mayoría de átomos estarían en la **superficie** (efectos de borde indeseados)
- El sistema no representaría correctamente un **sólido cristalino, líquido o membrana**

### Condiciones Periódicas de Contorno (PBC)

La solución es tratar la caja como si estuviera rodeada de **réplicas infinitas de sí misma**. Cuando una partícula sale por un lado, entra por el lado opuesto.

```
 ___________________________________________
| réplica | réplica | réplica | réplica ... |
|  (-1,-1) |  (0,-1) |  (1,-1) |  (2,-1)... |
|---------+---------+---------+-------------|
| réplica | CELDA   | réplica | réplica ... |
|  (-1, 0) | UNIDAD  |  (1, 0) |  (2, 0)... |
|          |  (0, 0) |         |             |
|---------+---------+---------+-------------|
| réplica | réplica | réplica | réplica ... |
|  (-1, 1) |  (0, 1) |  (1, 1) |  (2, 1)... |
 -------------------------------------------
```

In [ ]:
def aplicar_pbc(posicion, L):
    """
    Aplica condiciones periódicas de contorno (PBC) a una posición.
    Devuelve la posición dentro de la celda [0, L)^3.
    
    Args:
        posicion: array (3,) o (N, 3) de posiciones
        L: longitud de la caja cúbica (escalar o array de 3 elementos)
    
    Returns:
        Posición dentro de la celda
    """
    return posicion - L * np.floor(posicion / L)

def imagen_minima(dr, L):
    """
    Aplica la convención de imagen mínima al vector de desplazamiento.
    Garantiza que se use la réplica más cercana para calcular distancias.
    
    Args:
        dr: vector de desplazamiento (ri - rj)
        L: longitud de la caja cúbica
    
    Returns:
        Vector de desplazamiento corregido
    """
    return dr - L * np.round(dr / L)

# Demostración de PBC en 2D
np.random.seed(42)
L = 10.0  # caja de 10 Å
N = 20
pos = np.random.uniform(-5, 15, (N, 2))  # posiciones fuera de la caja
pos_pbc = aplicar_pbc(pos, L)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(pos[:, 0], pos[:, 1], c='red', s=80)
rect = patches.Rectangle((0, 0), L, L, linewidth=2, edgecolor='black', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_xlim(-6, 16)
axes[0].set_ylim(-6, 16)
axes[0].set_title('Posiciones originales (sin PBC)', fontsize=13)
axes[0].set_xlabel('x (Å)'); axes[0].set_ylabel('y (Å)')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(pos_pbc[:, 0], pos_pbc[:, 1], c='blue', s=80)
rect2 = patches.Rectangle((0, 0), L, L, linewidth=2, edgecolor='black', facecolor='lightyellow')
axes[1].add_patch(rect2)
axes[1].set_xlim(-1, L+1)
axes[1].set_ylim(-1, L+1)
axes[1].set_title('Posiciones con PBC aplicada', fontsize=13)
axes[1].set_xlabel('x (Å)'); axes[1].set_ylabel('y (Å)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pbc_demo.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Convención de Imagen Mínima y Radio de Corte

In [ ]:
# Visualizar imagen mínima
L = 10.0
r_i = np.array([1.0, 5.0])   # partícula i cerca del borde izquierdo
r_j = np.array([9.0, 5.0])   # partícula j cerca del borde derecho

dr_directo = r_j - r_i
dr_minima = imagen_minima(r_j - r_i, L)

print(f"Posición i: {r_i}")
print(f"Posición j: {r_j}")
print(f"Vector directo r_j - r_i: {dr_directo} → distancia = {np.linalg.norm(dr_directo):.2f} Å")
print(f"Imagen mínima:             {dr_minima} → distancia = {np.linalg.norm(dr_minima):.2f} Å")
print()
print("La imagen mínima da la distancia correcta entre partículas vecinas periódicas.")
print(f"La condición r_cut < L/2 = {L/2:.1f} Å garantiza que se usa solo una imagen.")

## 4. Tipos de Celda de Simulación

| Geometría | Descripción | Uso típico |
|-----------|-------------|------------|
| **Cúbica** | La más simple | Moléculas pequeñas, proteínas globulares |
| **Caja rectangular** | Tres lados distintos | Membranas lipídicas, nanoporos |
| **Dodecaedro rómbico** | ~29% menos volumen que la cúbica | Proteínas globulares (eficiente) |
| **Octaedro truncado** | Aún más esférica | Proteínas, reducción de artefactos |

La forma preferida para proteínas globulares es el **dodecaedro rómbico**, ya que:
- Minimiza el número de moléculas de agua necesarias
- Reduce los artefactos periódicos
- Ahorra hasta un 29% de tiempo de cómputo

## 5. Ensamble NVE (Microcanónico)

En el ensamble **NVE** (N partículas, Volumen y Energía constantes), no hay termostato ni barostato. La energía total se conserva exactamente.

**Usos:** pruebas de conservación de energía, validación del campo de fuerza.

In [ ]:
def calcular_temperatura_instante(velocidades, masas):
    """
    Calcula la temperatura instantánea a partir de las velocidades.
    T = sum(m_i * v_i^2) / (3 * N * k_B)
    
    Args:
        velocidades: array (N, 3) en m/s
        masas: array (N,) en kg
    
    Returns:
        Temperatura en K
    """
    N = len(masas)
    E_kin = 0.5 * np.sum(masas[:, np.newaxis] * velocidades**2)
    return 2 * E_kin / (3 * N * k_B)

# Simular fluctuaciones de temperatura en un gas ideal (NVE)
np.random.seed(0)
N = 100
T_target = 300.0  # K
masa_ar = 39.948 * 1.66054e-27  # kg
masas = np.full(N, masa_ar)

# Velocidades iniciales
sigma_v = np.sqrt(k_B * T_target / masa_ar)
velocidades = np.random.normal(0, sigma_v, (N, 3))
velocidades -= velocidades.mean(axis=0)

# Simular fluctuaciones de T en NVE (muestreo de configuraciones)
temperaturas_nve = []
for _ in range(500):
    # En NVE real las velocidades cambian continuamente; aquí muestreamos distribuciones
    v_sample = np.random.normal(0, sigma_v, (N, 3))
    v_sample -= v_sample.mean(axis=0)
    # Reescalar para conservar E
    T_inst = calcular_temperatura_instante(v_sample, masas)
    temperaturas_nve.append(T_inst)

T_arr = np.array(temperaturas_nve)

plt.figure(figsize=(10, 4))
plt.plot(T_arr, 'b-', linewidth=0.8, alpha=0.8)
plt.axhline(T_target, color='r', linestyle='--', label=f'T objetivo = {T_target} K')
plt.axhline(T_arr.mean(), color='g', linestyle='--', label=f'T promedio = {T_arr.mean():.1f} K')
plt.xlabel('Paso de muestreo', fontsize=12)
plt.ylabel('Temperatura (K)', fontsize=12)
plt.title(f'Fluctuaciones de Temperatura en NVE - N={N} partículas', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fluctuaciones_T_NVE.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"T media = {T_arr.mean():.2f} K, σ(T) = {T_arr.std():.2f} K")
print(f"Fluctuación relativa = {T_arr.std()/T_arr.mean()*100:.2f}%")

## 6. Ensamble NVT (Canónico) y el Reescalado de Velocidades

Para controlar la temperatura en DM se usan **termostatos**. El más simple es el **reescalado de velocidades**:

$$\mathbf{v}_i' = \lambda \mathbf{v}_i, \qquad \lambda = \sqrt{\frac{T_{\text{objetivo}}}{T_{\text{actual}}}}$$

Sin embargo, el reescalado directo no genera el ensamble NVT correcto. Los termostatos modernos más utilizados son:
- **v-rescale** (GROMACS): reescalado estocástico, genera NVT correcto
- **Nosé-Hoover**: baño térmico extendido, conserva volumen en espacio de fases
- **Langevin**: fricción + fuerzas aleatorias, popular en AMBER y NAMD

In [ ]:
def termostato_reescalado(velocidades, masas, T_objetivo):
    """
    Termostato de reescalado simple de velocidades.
    
    Args:
        velocidades: array (N, 3) en m/s
        masas: array (N,) en kg
        T_objetivo: temperatura objetivo en K
    
    Returns:
        Velocidades reescaladas
    """
    T_curr = calcular_temperatura_instante(velocidades, masas)
    if T_curr < 1e-10:
        return velocidades
    lamda = np.sqrt(T_objetivo / T_curr)
    return velocidades * lamda

def termostato_berendsen(velocidades, masas, T_objetivo, dt, tau_T=0.1e-12):
    """
    Termostato de Berendsen: acoplamiento débil a un baño.
    τ_T: constante de tiempo del acoplamiento (típicamente 0.1-1 ps)
    """
    T_curr = calcular_temperatura_instante(velocidades, masas)
    if T_curr < 1e-10:
        return velocidades
    lamda = np.sqrt(1 + (dt / tau_T) * (T_objetivo / T_curr - 1))
    return velocidades * lamda

# Simular un sistema que se calienta y es controlado por termostato
np.random.seed(123)
T_inicial = 200.0  # K
T_obj = 300.0      # K
N = 100
dt = 2e-15  # 2 fs
sigma_v_ini = np.sqrt(k_B * T_inicial / masa_ar)
vel = np.random.normal(0, sigma_v_ini, (N, 3))
vel -= vel.mean(axis=0)

T_rescalado = [calcular_temperatura_instante(vel, masas)]
vel_B = vel.copy()
T_berendsen = [calcular_temperatura_instante(vel_B, masas)]

n_steps = 500
for _ in range(n_steps):
    # Pequeña perturbación para simular evolución dinámica
    dv = np.random.normal(0, sigma_v * 0.01, (N, 3))
    vel = vel + dv
    vel_B = vel_B + dv

    # Aplicar termostatos
    vel = termostato_reescalado(vel, masas, T_obj)
    vel_B = termostato_berendsen(vel_B, masas, T_obj, dt, tau_T=100 * dt)

    T_rescalado.append(calcular_temperatura_instante(vel, masas))
    T_berendsen.append(calcular_temperatura_instante(vel_B, masas))

pasos = np.arange(n_steps + 1)
plt.figure(figsize=(10, 5))
plt.plot(pasos, T_rescalado, 'b-',  label='Reescalado simple',  linewidth=1.5)
plt.plot(pasos, T_berendsen, 'g--', label='Berendsen',          linewidth=1.5)
plt.axhline(T_obj, color='r', linestyle=':', label=f'T objetivo = {T_obj} K', linewidth=2)
plt.xlabel('Paso', fontsize=12)
plt.ylabel('Temperatura (K)', fontsize=12)
plt.title('Termostatización: Reescalado vs Berendsen', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('termostatos.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Ensamble NPT y Barostatos

Para simular condiciones fisiológicas (1 bar, 300 K) se usa el ensamble **NPT** con un **barostato** que controla la presión escalando el volumen de la caja.

La presión instantánea se calcula con el **teorema del virial**:

$$P = \frac{N k_B T}{V} + \frac{1}{3V} \sum_{i<j} \mathbf{r}_{ij} \cdot \mathbf{F}_{ij}$$

Los barostatos más usados son:
- **Berendsen**: escalado isótropo o anisotrópico del volumen
- **Parrinello-Rahman**: barostato de masa extendida, correcto para NPT
- **Monte Carlo barostat** (OpenMM): cambios de volumen estocásticos

| Barostato | Ensamble correcto | Software | Notas |
|-----------|-------------------|----------|-------|
| Berendsen | No exactamente | GROMACS | Bueno para equilibración |
| Parrinello-Rahman | Sí | GROMACS | Para producción NPT |
| MC Barostat | Sí | OpenMM, AMBER | Simple y correcto |

## 8. Selección del Ensamble Según el Problema

```
¿Qué quiero simular?
├── Validar campo de fuerza / conservación energía  → NVE
├── Equilibración del sistema                       → NVT
├── Simulación de producción en solución            → NPT
├── Membrana lipídica                               → NPT (semianisótropo)
├── Cristal                                         → NPT (anisótropo)
└── Proteína aislada en vacío                       → NVT
```

**Protocolo típico para una proteína en solución:**
1. Minimización de energía (sin restricciones de ensamble)
2. NVT: 0.1-1 ns calentando de 0 K a 300 K
3. NPT: 0.1-1 ns ajustando densidad del sistema
4. NPT: producción (10-100+ ns)

In [ ]:
# Visualización de los ensambles estadísticos
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

np.random.seed(42)
n_frames = 300
t_sim = np.arange(n_frames)

# NVE: energía total constante, T fluctúa
E_total_nve = 100 + np.random.normal(0, 0.1, n_frames)
T_nve = 300 + np.random.normal(0, 8, n_frames)
axes[0].plot(t_sim, T_nve, 'b-', linewidth=1, alpha=0.8)
axes[0].axhline(300, color='r', linestyle='--', linewidth=1.5, label='T=300K')
axes[0].set_title('NVE - T fluctúa libremente', fontsize=12)
axes[0].set_xlabel('Paso'); axes[0].set_ylabel('T (K)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# NVT: T controlada, V constante
T_nvt = 300 + np.random.normal(0, 1.5, n_frames)
axes[1].plot(t_sim, T_nvt, 'g-', linewidth=1, alpha=0.8)
axes[1].axhline(300, color='r', linestyle='--', linewidth=1.5, label='T=300K')
axes[1].set_title('NVT - T controlada por termostato', fontsize=12)
axes[1].set_xlabel('Paso'); axes[1].set_ylabel('T (K)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# NPT: T y P controladas
V_npt = 100 + np.cumsum(np.random.normal(0, 0.05, n_frames))
V_npt = V_npt - V_npt.mean() + 100  # centrar alrededor de 100
axes[2].plot(t_sim, V_npt, 'm-', linewidth=1, alpha=0.8)
axes[2].axhline(100, color='r', linestyle='--', linewidth=1.5, label='V₀=100 nm³')
axes[2].set_title('NPT - Volumen fluctúa (P controlada)', fontsize=12)
axes[2].set_xlabel('Paso'); axes[2].set_ylabel('Volumen (nm³)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ensambles.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Ejercicios

### Ejercicio 1 (Básico)
Implementa la función `imagen_minima` para una caja no cúbica (tres longitudes diferentes: Lx, Ly, Lz). Verifica que la distancia entre partículas se calcula correctamente para ambos lados de la caja.

### Ejercicio 2 (Intermedio)
Implementa el termostato de Nosé-Hoover simplificado y compara las distribuciones de temperatura que genera con el termostato de Berendsen. ¿Cuál genera correctamente la distribución de Maxwell-Boltzmann?

### Ejercicio 3 (Avanzado)
Implementa un gas de Lennard-Jones de N=50 partículas en 2D con PBC. Calcula la presión instantánea usando el teorema del virial y compárala con la ecuación de estado del gas ideal $P = Nk_BT/V$.

In [ ]:
# Ejercicio 1 - Imagen mínima para caja no cúbica
def imagen_minima_ortorhombica(dr, Lx, Ly, Lz):
    """
    Convención de imagen mínima para caja ortorómbica.
    
    Args:
        dr: vector de desplazamiento (dx, dy, dz)
        Lx, Ly, Lz: dimensiones de la caja
    
    Returns:
        Vector corregido
    """
    L = np.array([Lx, Ly, Lz])
    return dr - L * np.round(dr / L)

# Prueba
Lx, Ly, Lz = 10.0, 8.0, 12.0
r_i = np.array([0.5, 0.5, 0.5])
r_j = np.array([9.5, 7.5, 11.5])
dr = r_j - r_i
dr_corr = imagen_minima_ortorhombica(dr, Lx, Ly, Lz)

print(f"Caja: {Lx} x {Ly} x {Lz} Å")
print(f"Vector directo:   {dr}, |dr| = {np.linalg.norm(dr):.3f} Å")
print(f"Imagen mínima:    {dr_corr}, |dr| = {np.linalg.norm(dr_corr):.3f} Å")

## 10. Recursos Adicionales

- **Libros:**
  - Allen & Tildesley, *Computer Simulation of Liquids*, Cap. 1-3 (2017)
  - Tuckerman, *Statistical Mechanics*, Cap. 4-5 (2010)

- **Documentación:**
  - [GROMACS - Periodic Boundary Conditions](https://manual.gromacs.org/documentation/current/reference-manual/algorithms/periodic-boundary-conditions.html)
  - [GROMACS - Temperature Coupling](https://manual.gromacs.org/documentation/current/reference-manual/algorithms/temperature-coupling.html)
  - [GROMACS - Pressure Coupling](https://manual.gromacs.org/documentation/current/reference-manual/algorithms/pressure-coupling.html)

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Implementar condiciones periódicas de contorno (PBC) en 2D y 3D
- ✅ Aplicar la convención de imagen mínima para el cálculo de distancias
- ✅ Distinguir los ensambles NVE, NVT y NPT y sus aplicaciones
- ✅ Calcular temperatura y presión instantánea a partir de una trayectoria
- ✅ Implementar un termostato de Berendsen simple en Python

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 5.3: Condiciones de Contorno y Ensambles Estadísticos**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_5.2-Integradores_y_Algoritmos-blue.svg)](02_integradores_algoritmos.ipynb)
[![Siguiente](https://img.shields.io/badge/Actividad_5.4_➡️-Termostatos_y_Barostatos-green.svg)](04_termostatos_barostatos.ipynb)

---

📚 **[Volver al Módulo 5](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>